# Основы pandas

В этом разделе мы познакомимся с библиотекой `pandas`: научимся читать табличные данные, исследовать DataFrame, выбирать строки и столбцы, выполнять вычисления, фильтровать данные, группировать их и автоматизировать анализ нескольких файлов.

`pandas` — главная библиотека Python для работы с табличными данными. Она умеет читать данные из CSV, Excel, JSON, SQL и многих других форматов.


## 0. Импортируем библиотеку


In [ ]:
import pandas as pd

## 1. Основные структуры данных

В pandas две ключевые структуры:

- **DataFrame** — таблица с именованными строками и столбцами (как электронная таблица)
- **Series** — одномерная последовательность значений (один столбец или строка DataFrame)

Оба объекта имеют **индекс** — метки строк, по которым можно обращаться к данным.


## 2. Чтение данных из файла


In [ ]:
# Чтение CSV-файла
data = pd.read_csv("data/temp_Jan1_2020.csv")

### Дополнительные параметры


In [ ]:
# Если нужно пропустить строки заголовка
# data = pd.read_csv("data/stations.csv", skiprows=8)

# Если разделитель не запятая, а точка с запятой:
# data = pd.read_csv("data/stations.csv", sep=";")

# Читать только нужные столбцы:
# data = pd.read_csv("data/stations.csv", usecols=["NAME", "LAT", "LON"])

## 3. Первый взгляд на данные


In [ ]:
data.head()     

In [ ]:
data.tail(3) # последние 3 строки

In [ ]:
data.shape # (строк, столбцов)

In [ ]:
data.dtypes # типы данных по столбцам

In [ ]:
data.info() # сводная информация

In [ ]:
data.describe()  # описательная статистика

In [ ]:
print(data.columns.values) # Имена столбцов

In [ ]:
print(len(data)) # Количество строк

## 4. Выбор столбцов


In [ ]:
# Один столбец → Series
temps = data["AvgTemperature"]

temps

In [ ]:
# Несколько столбцов → DataFrame
subset = data[["City", "AvgTemperature"]]

subset.head()

## 5. Базовая статистика


In [ ]:
print(data["AvgTemperature"].mean())    # среднее
print(data["AvgTemperature"].min())     # минимум
print(data["AvgTemperature"].max())     # максимум
print(data["AvgTemperature"].std())     # стандартное отклонение
print(data["AvgTemperature"].nunique()) # количество уникальных значений

## 6. Создание DataFrame вручную

Иногда данные удобно создать прямо в коде — из словаря или из списков.


In [ ]:
# Из словаря
stations = pd.DataFrame({
    "name":  ["Москва", "Казань", "Новосибирск"],
    "lat":   [55.7522,  55.7963,  54.8480],
    "lon":   [37.6156,  49.1088,  83.1048],
    "elev":  [144,      116,      162],
})

stations

In [ ]:
# Из списка словарей (типичная структура при чтении JSON)
records = [
    {"name": "Москва",      "lat": 55.7522, "lon": 37.6156},
    {"name": "Казань",      "lat": 55.7963, "lon": 49.1088},
    {"name": "Новосибирск", "lat": 54.8480, "lon": 83.1048},
]

df = pd.DataFrame(records)
df

## 7. Сохранение данных


In [ ]:
# # Сохранить в CSV
# data.to_csv("output.csv")

# Сохранить в Excel
# data.to_excel("output.xlsx", index=False)

## 8. Вычисления и новые столбцы

В pandas вычисления выполняются **векторизованно** — сразу для всех строк, без цикла. Это быстро и читаемо.


In [ ]:
# Перевод в градусы Цельсия
data['AvgTemperature_c'] = (data['AvgTemperature'] - 32) * 5 / 9

data.head()

## 9. Выборка строк и столбцов

### `.loc[]` — выборка по индексам


In [ ]:
# Строки 0–5, один столбец
data.loc[0:5, "AvgTemperature_c"]


In [ ]:
# Строки 0–5, несколько столбцов
data.loc[0:5, ["AvgTemperature_c", "AvgTemperature"]]

> **Важно:** `loc[0:5]` возвращает **6** строк (0, 1, 2, 3, 4, 5) — включительно с обеих сторон.


In [ ]:
# Одно значение
data.loc[0, "AvgTemperature_c"]

### `.iloc[]` — выборка по позиции


In [ ]:
# Строки 0–4, столбцы 0–1 (по позиции, не включая конец)
data.iloc[0:5, 0:2]

In [ ]:
# Последняя строка, последний столбец
data.iloc[-1, -1]

## 10. Фильтрация по условию


In [ ]:
hot_cities = data.loc[data["AvgTemperature_c"] >= 25]

hot_cities

In [ ]:
# Несколько условий — используем & (и) и | (или)
hot_cities_asia = data.loc[(data["AvgTemperature_c"] >= 25) & (data["Region"] == 'Asia')]

hot_cities_asia

In [ ]:
# Выбор по списку значений
hot_cities_asia_africa = data.loc[data["Region"].isin(['Asia', 'Africa']) & (data["AvgTemperature_c"] >= 25)]

hot_cities_asia_africa

При фильтрации индекс сохраняется из оригинала. Чтобы сбросить — используйте `.reset_index(drop=True)`. Также рекомендуется делать `.copy()`, чтобы не изменять исходный DataFrame случайно:


In [ ]:
hot_cities_asia = data.loc[(data["AvgTemperature_c"] >= 25) & (data["Region"] == 'Asia')].copy()
hot_cities_asia = hot_cities_asia.reset_index(drop=True)
hot_cities_asia.head()

## 11. Сортировка


In [ ]:
# По возрастанию
data.sort_values(by="AvgTemperature_c").head()

In [ ]:
# По убыванию
data.sort_values(by="AvgTemperature_c", ascending=False).head()

## 14. Переименование столбцов


In [ ]:
data = data.rename(columns={
    "AvgTemperature": "t_Fahrenheit_Jan12020",
    "AvgTemperature_c": "t_Celsius_Jan12020",
})

print(data.columns.values)

## 15. Объединение таблиц (join)

Объединение двух DataFrame по общему ключевому столбцу — частая операция при работе с несколькими источниками данных.


In [ ]:
# Чтение CSV-файла
data1997 = pd.read_csv("data/temp_Jan1_1997.csv")

# # Ключ называется одинаково в обоих таблицах
result = data.merge(data1997, on='City')

result.head()



Есть две потенциальные проблемы:

1. `City` может быть **не уникальным значением**
   Например, города с одинаковым названием могут находиться в разных странах.

2. Названия городов могут **отличаться между таблицами**
   Например: `Moscow` vs `Moskva`, различия в регистре, пробелах или вариантах написания.

В нашем случае второй проблемы не возникнет, так как оба датасета получены из одного источника, и названия городов записаны одинаково. Однако при работе с реальными данными это всегда нужно проверять заранее.

> В задачах объединения таблиц лучше использовать не названия, а специальные уникальные идентификаторы (`id`), потому что текстовые поля могут содержать ошибки, дубликаты и разные варианты записи.


Сначала проверим количество строк и уникальных городов:


In [ ]:
print("data rows:", len(data))
print("data unique cities:", data['City'].nunique())

Проверим, какие города повторяются:


In [ ]:
data['City'].value_counts()[data['City'].value_counts() > 1]

Потом лучше объединять не только по городу, а по городу и стране.

Также перед объединением таблиц полезно заранее отфильтровывать ненужные данные. Например, если для анализа нам нужны только столицы, остальные города можно сразу исключить. Это упростит объединение, уменьшит количество ошибок и сделает работу с данными быстрее.


In [ ]:
result = data.merge(
    data1997[['City', 'Country', 'AvgTemperature']],
    on=['City', 'Country'],
    how='left'
)

result

Переназавём присоединенный столбец. Перед дальнейшим анализом полезно давать столбцам более понятные и информативные названия. Например, название AvgTemperature не говорит нам, для какой даты рассчитана температура и в каких единицах она измеряется.


In [ ]:
result= data.rename(columns={
    "AvgTemperature": "t_Fahrenheit_Jan11997"
})

## 16. Группировка и агрегация (`.groupby()`)

`.groupby()` разбивает DataFrame на группы по уникальным значениям одного (или нескольких) столбцов, затем применяет агрегирующую функцию к каждой группе.

Посчитать среднюю температуру по регионам мира 1 января 2020 года.


In [ ]:
# Средняя температура по регионам
region_temp = (
    data.groupby("Region")["t_Celsius_Jan12020"]
    .mean()
    .reset_index()
)

region_temp

Также посчитаем среднюю температуру по странам 1 января 2020 года


In [ ]:
# Средняя температура по странам
country_temp = (
    data.groupby("Country")["t_Celsius_Jan12020"]
    .mean()
    .reset_index()
)

country_temp

> Материалы частично основаны на курсе:
>
> [_Python for Geographic Data Analysis_](https://pythongis.org/part1/index.html) (Henrikki Tenkanen, Vuokko Heikinheimo, David Whipp).
